In [ ]:
# The Google Earth Engine module
import ee

# Geopandas allows us to read the shapefile used to
# define the region of interest (ROI)
import geopandas

In [ ]:
ee_prj_name = "ee-pb-dev"  # <==== Replace this with your own EE project string
ee.Authenticate()
ee.Initialize(project=ee_prj_name)

In [ ]:
# The region of interest
vec_tiles_file = "roi_5deg_grid.geojson"
tiles_gdf = geopandas.read_file(vec_tiles_file)

# Define the target bands
target_bands = ['treecover2000', 'loss', 'gain', 'lossyear', 'datamask']

# Output directory on Google Drive
out_gdrive_dir = "hansen_forest_change_data"

In [ ]:

# Iterate through each row (i.e. Tile) and export the data for that tile.
for index, row in tiles_gdf.iterrows():
    tile_name = row['tile_names']
    print(f"Tile: {tile_name}")

    # Get layer bbox: minx, miny, maxx, maxy
    minx, miny, maxx, maxy = row['geometry'].bounds
    gee_bbox_aoi = ee.Geometry.BBox(minx, miny, maxx, maxy)

    gfc_hansen_img = ee.Image("UMD/hansen/global_forest_change_2025_v1_13").clip(gee_bbox_aoi).select(target_bands)

    out_file_name = f"{tile_name}_hansen_forest_chng"

    # Export the Hansen Forest Change image to your Google Drive
    task = ee.batch.Export.image.toDrive(
        image=gfc_hansen_img,
        description=out_file_name,
        folder=out_gdrive_dir,
        scale=30,
        region=gee_bbox_aoi,
        maxPixels=1e13,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True, "noData": 0.0},
    )
    task.start()
